## NBA 3.1 | Additional (add-on) Patient Features — Skill Section 13

*Which HCPs have the most addressable Kerendia-eligible patients, recent trigger events and switch candidates, and which face barriers to starting?*

**Sources:** Rx (`rx_all`, paid claims `CLAIM_STATUS = 'S'`), Dx (`dx_full_history`), Lab (`lab`), Px (`px_all`).

**Method (Section 13.0):**
1. Patient-level status from claims by **any** provider (not filtered to the HCP universe). Each building block becomes a 28-bit mask (bit *i* = cohort *i*), so all 28 cohorts are evaluated in one pass and combined with bitwise AND / OR / NOT.
2. HCP panel = distinct patients with any paid Rx, Dx, lab or albuminuria-test (CPT 82043/82044) claim from the HCP in the lookback. **Only HCP-universe HCPs get a panel.**
3. Roll-up: `NUM_OF_PATIENTS_{stem}` and `PCT_OF_PATIENTS_{stem}` (= NUM / panel). Triggers (13.2) and MRA (13.5) are credited to the HCP on the claim. `PCT_*` / `RATIO_*` stay null when the denominator is 0.

| Section | Features |
| --- | --- |
| 13.1 Untreated eligible pool | 8 NUM+PCT stems + `PCT_OF_DIABETIC_CKD_..._UNTREATED_ELIGIBLE` + `RATIO_KERENDIA_TREATED_TO_ELIGIBLE` |
| 13.2 90-day triggers | new DKD Dx, new SGLT2 start, new albuminuria test (NUM+PCT) |
| 13.3 HF with LVEF >= 40% proxy | 4 NUM+PCT stems |
| 13.4 Potassium barrier | hyperkalemia among DKD patients (NUM+PCT of DKD); potassium-binder stem built only if Lokelma/Veltassa are in the Rx event list |
| 13.5 MRA switch | current / stopped MRA with T2D+CKD (NUM+PCT), % of HCP's MRA patients who stopped |

**Data gaps found in the 3.1 inputs (2026-11-02):** `E87.5` is not in the Dx event list (hyperkalemia uses potassium > 5.0 mEq/L from lab only), Lokelma/Veltassa are not in the Rx event list (binder stem skipped), CPT 82044 is not in the Px event list (82043 + UACR lab only). The 3.1 Rx event list has product names, not class names, so classes are mapped by product name (see `rx_class_groups`, printed at run time).

**Output:** `.../kerendia_nba_3_1_volume/notebook_output/patient_features/addon/claims_addon_patient_features` (grain `HCP_COHORT_ID`, partitioned by `COHORT`).

**Validation (300-HCP sample, cohort c14):** panel size, diabetic CKD, untreated eligible, new-SGLT2 trigger and MRA-stopped % matched an independent pandas reference exactly (0 mismatches).

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/02_Input_Data_Creation/00_event_list_config"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/00_Helper_Notebooks/00_cohort_generation"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/03_patient_features/Final Notebooks/00_Patient_Feature_Helpers_31"

In [0]:
from pyspark.sql import functions as F

FEATURE_FAMILY = "claims"

DX_FULL_PATH = INPUT_PATHS_31["dx_full_history"]
RX_ALL_PATH  = INPUT_PATHS_31["rx_all"]
LAB_PATH     = INPUT_PATHS_31["lab"]
PX_ALL_PATH  = INPUT_PATHS_31["px_all"]

OUTPUT_BASE = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
               "kerendia_nba_3/kerendia_nba_3_1_volume/notebook_output/patient_features/addon")
ADDON_OUT   = f"{OUTPUT_BASE}/claims_addon_patient_features"
TMP_BASE    = f"{OUTPUT_BASE}/_tmp"

dbutils.widgets.text("TEST_HCP_SAMPLE", "0", "HCP sample size (0=full)")
TEST_HCP_SAMPLE = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))
if TEST_HCP_SAMPLE > 0:   # sample runs write to their own folder so full-universe outputs are never overwritten
    OUTPUT_BASE = f"{OUTPUT_BASE}/_test_sample_{TEST_HCP_SAMPLE}"
    ADDON_OUT   = f"{OUTPUT_BASE}/claims_addon_patient_features"
    TMP_BASE    = f"{OUTPUT_BASE}/_tmp"

# Section 13.0 parameters
ADDON_PARAMS = dict(
    ON_THERAPY_RECENT_DAYS = 180,   # window for "currently on" therapy flags
    TRIGGER_WINDOW_DAYS    = 90,    # window for trigger events
    NEW_EVENT_WASHOUT_DAYS = 365,   # washout for "new" event detection
    MRA_CURRENT_GRACE_DAYS = 30,    # MRA current: last fill end >= lookback end - 30 d
    MRA_STOP_GAP_DAYS      = 60,    # MRA stopped: last fill end + 60 d < lookback end
    COHORT_BATCH           = 7,     # cohorts per aggregation pass in the HCP roll-up
    TMP                    = TMP_BASE,
)
print(f"ADDON_OUT: {ADDON_OUT}")

In [0]:
cohorts = CohortGenerator(FEATURE_FAMILY).load()
meta = cohorts.attrs["metadata"]
assert meta["feature_family"] == "claims" and meta["offset_days"] == 0
assert list(cohorts["cohort_id"]) == [f"c{i}" for i in range(1, 29)]
assert (str(cohorts.iloc[0]["lookback_start_date"]), str(cohorts.iloc[0]["lookback_end_date"])) == ("2023-06-01", "2024-05-01")

HCP_UNIVERSE_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
                     "kerendia_nba_3/kerendia_nba_3_1_volume/HCP_Universe_without_call_hcps")
hcp_universe_df = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()
if TEST_HCP_SAMPLE > 0:
    hcp_universe_df = hcp_universe_df.orderBy(F.xxhash64("PROVIDER_ID"), "PROVIDER_ID").limit(TEST_HCP_SAMPLE)
hcp_universe = F.broadcast(hcp_universe_df)
rx_event_keys = event_groups("rx")
print(f"Cohorts: {len(cohorts)} | Rx EVENT_GROUPs: {len(rx_event_keys)}")

In [0]:
import builtins, re
from datetime import timedelta as _td
from functools import reduce

# ---------------------------------------------------------------- 13.0 building-block definitions
DX_BLOCKS = {   # block -> SQL predicate on DIAGNOSIS_CODE (codes carry dots, e.g. 'E11.22')
    "T2D":     "DIAGNOSIS_CODE LIKE 'E11%'",
    "DKD":     "DIAGNOSIS_CODE IN ('E11.21','E11.22','E11.29')",
    "CKD":     "DIAGNOSIS_CODE IN ('N18.1','N18.2','N18.3','N18.30','N18.31','N18.32','N18.4','N18.5','N18.9',"
               "'I12.9','I13.0','I13.10','E11.21','E11.22','E11.29','E08.21')",
    "ESRD":    "DIAGNOSIS_CODE IN ('N18.6','I12.0','I13.11','I13.2','Z99.2') OR DIAGNOSIS_CODE LIKE 'Z49%'",
    "ALB_DX":  "DIAGNOSIS_CODE LIKE 'R80%'",
    "HFPEF":   "DIAGNOSIS_CODE LIKE 'I50.3%'",
    "HFREF":   "DIAGNOSIS_CODE LIKE 'I50.2%'",
    "HFCOMB":  "DIAGNOSIS_CODE LIKE 'I50.4%'",
    "HYPERK":  "DIAGNOSIS_CODE = 'E87.5'",
}

def rx_class_groups(rx_groups):
    """Map the 3.1 Rx EVENT_GROUPs (product names, no class names) to the Section 13.0 classes."""
    g = [x for x in rx_groups if isinstance(x, str)]
    u = lambda pat, excl=(): sorted(x for x in g if re.search(pat, x.upper()) and x.upper() not in excl)
    return {
        # ACE inhibitors + ARBs and their combos (ARNI Entresto / sacubitril-valsartan excluded: separate class)
        "ACEARB":  u(r"PRIL\b|PRIL |PRIL/|SARTAN", excl=("ENTRESTO", "SACUBITRIL/VALSARTAN")),
        "SGLT2":   u(r"FLOZIN|JARDIANCE|FARXIGA|INVOKANA|INVOKAMET|STEGLATRO|STEGLUJAN|INPEFA|GLYXAMBI|SEGLUROMET|"
                     r"SYNJARDY|XIGDUO|QTERN|TRIJARDY|BRENZAVVY"),
        "GLP1":    u(r"OZEMPIC|RYBELSUS|TRULICITY|VICTOZA|MOUNJARO|LIRAGLUTIDE|BYETTA|BYDUREON|SOLIQUA|XULTOPHY|"
                     r"WEGOVY|SAXENDA|ZEPBOUND"),
        "MRA":     u(r"SPIRONOLACTONE|ALDACTONE|CAROSPIR|EPLERENONE|INSPRA"),
        "KER":     u(r"^KERENDIA$"),
        "LOOP":    u(r"FUROSEMIDE|BUMETANIDE|TORSEMIDE|LASIX|BUMEX|DEMADEX|SOAANZ"),
        "KBINDER": u(r"LOKELMA|VELTASSA|PATIROMER|ZIRCONIUM"),
    }

def build_block_events(dx_full, rx_all, lab, px_all, rx_classes, max_date):
    """One long table of building-block events from ALL providers (patient status is provider-independent).
    Columns: PROVIDER_ID, PATIENT_ID, BLOCK, D (date), E (fill end, Rx), V (lab value)."""
    cols = ["PROVIDER_ID", "PATIENT_ID", "BLOCK", "D", "E", "V"]
    parts = []
    dx = dx_full.select("PROVIDER_ID", "PATIENT_ID", "DIAGNOSIS_CODE", F.to_date("SVC_DT").alias("D"))
    dx_blocks = F.array(*[F.when(F.expr(p), F.lit(b)) for b, p in DX_BLOCKS.items()])
    parts.append(dx.select("PROVIDER_ID", "PATIENT_ID", F.explode(F.array_compact(dx_blocks)).alias("BLOCK"), "D",
                           F.lit(None).cast("date").alias("E"), F.lit(None).cast("double").alias("V")))
    # Rx: paid claims only (CLAIM_STATUS = 'S'); days supply missing or > 180 -> 30; fill end = SVC_DT + DS - 1
    g2b = [(grp, b) for b, gs in rx_classes.items() for grp in gs]
    rmap = F.broadcast(spark.createDataFrame(g2b, "EVENT_NAME string, BLOCK string"))
    ds = F.expr("try_cast(DAYS_SUPPLY AS DOUBLE)").cast("int")   # stored as string '30.000000'
    ds = F.when(ds.isNull() | (ds <= 0) | (ds > 180), F.lit(30)).otherwise(ds)
    parts.append(rx_all.filter(F.col("CLAIM_STATUS") == "S").join(rmap, "EVENT_NAME")
                       .select("PROVIDER_ID", "PATIENT_ID", "BLOCK", F.to_date("SVC_DT").alias("D"),
                               F.date_add(F.to_date("SVC_DT"), ds - 1).alias("E"), F.lit(None).cast("double").alias("V")))
    # Lab: latest-value blocks with plausibility filters (eGFR 1-200, UACR 0-20,000, K+ 1-10)
    v = F.col("MEDIAN_TEST_RESULT")
    lab_b = (F.when((F.col("TEST_TYPE") == "UACR") & v.between(0, 20000), F.lit("UACR"))
              .when((F.col("TEST_TYPE") == "GFR") & v.between(1, 200), F.lit("EGFR"))
              .when(F.col("LOINC_CD").isin("2823-3", "6298-4") & v.between(1, 10), F.lit("K")))
    labx = lab.select("PROVIDER_ID", "PATIENT_ID", lab_b.alias("BLOCK"), F.to_date("SVC_DT").alias("D"),
                      F.lit(None).cast("date").alias("E"), v.alias("V")).filter(F.col("BLOCK").isNotNull())
    parts.append(labx)
    # Albuminuria test = UACR lab result or CPT 82043/82044; echo tests from the Px event list
    parts.append(labx.filter(F.col("BLOCK") == "UACR").withColumn("BLOCK", F.lit("ALBTEST"))
                     .withColumn("V", F.lit(None).cast("double")))
    pxb = (F.when(F.col("PROCEDURE_CODE").isin("82043", "82044"), F.lit("ALBTEST"))
            .when(F.col("EVENT_NAME") == "echo_tests", F.lit("ECHO")))
    parts.append(px_all.select("PROVIDER_ID", "PATIENT_ID", pxb.alias("BLOCK"), F.to_date("SVC_DT").alias("D"),
                               F.lit(None).cast("date").alias("E"), F.lit(None).cast("double").alias("V"))
                       .filter(F.col("BLOCK").isNotNull()))
    ev = reduce(lambda a, b: a.unionByName(b), [p.select(*cols) for p in parts])
    return ev.filter(F.col("PATIENT_ID").isNotNull() & (F.col("D") <= F.lit(max_date)))

# ---------------------------------------------------------------- per-cohort 28-bit masks
def _mask(rows, cond_fn):
    """BIGINT with bit i set when cond_fn(cohort_row) is true for cohort i."""
    return reduce(lambda a, b: a + b,
                  [F.when(cond_fn(r), F.lit(1 << i)).otherwise(F.lit(0)).cast("long") for i, r in enumerate(rows)])

def _win(r, kind, p):
    L = r["LAST_DAY"]
    if kind == "LB":   return r["START"], L
    if kind == "R180": return L - _td(days=p["ON_THERAPY_RECENT_DAYS"] - 1), L
    if kind == "D90":  return L - _td(days=p["TRIGGER_WINDOW_DAYS"] - 1), L
    if kind == "WASH": return (L - _td(days=p["TRIGGER_WINDOW_DAYS"] - 1 + p["NEW_EVENT_WASHOUT_DAYS"]),
                               L - _td(days=p["TRIGGER_WINDOW_DAYS"]))
    raise ValueError(kind)

def lambda_exists(a, b):
    return F.exists("EV", lambda x: x["D"].between(F.lit(a), F.lit(b)))

PATIENT_MASKS = {   # block -> mask kinds
    "T2D": ["LB"], "DKD": ["LB", "D90", "BEF90"], "CKD": ["LB"], "ESRD": ["LB"], "ALB_DX": ["LB"],
    "HFPEF": ["LB"], "HFREF": ["LB"], "HFCOMB": ["LB"], "HYPERK": ["LB"],
    "ACEARB": ["R180"], "SGLT2": ["R180", "D90", "WASH"], "GLP1": ["R180"], "LOOP": ["R180"], "KBINDER": ["R180"],
    "MRA": ["R180", "CUR", "STOP"], "KER": ["LB", "R180", "EVER"],
    "UACR": ["GE30"], "EGFR": ["GE25"], "K": ["GT5"], "ALBTEST": ["D90", "WASH"], "ECHO": ["LB"],
}

def build_patient_flags(ev, rows, p):
    """Patient-level status: one row per patient, one BIGINT mask column per BLOCK_KIND (bit i = cohort i)."""
    pb = (ev.groupBy("PATIENT_ID", "BLOCK", "D").agg(F.max("E").alias("E"), F.max("V").alias("V"))
            .groupBy("PATIENT_ID", "BLOCK").agg(F.collect_list(F.struct("D", "E", "V")).alias("EV")))
    pb = _checkpoint(pb, p.get("TMP") and f"{p['TMP']}/_addon_pb")
    longs = []
    for block, kinds in PATIENT_MASKS.items():
        sub = pb.filter(F.col("BLOCK") == block)
        exprs = []
        for k in kinds:
            if k in ("LB", "R180", "D90", "WASH"):
                m = _mask(rows, lambda r, k=k: lambda_exists(*_win(r, k, p)))
            elif k == "BEF90":     # any earlier code in history (for "newly diagnosed")
                m = _mask(rows, lambda r: F.exists("EV", lambda x: x["D"] < F.lit(_win(r, "D90", p)[0])))
            elif k == "EVER":      # full history up to LOOKBACK_END (never on Kerendia)
                m = _mask(rows, lambda r: F.exists("EV", lambda x: x["D"] <= F.lit(r["LAST_DAY"])))
            elif k in ("GE30", "GE25", "GT5"):   # latest lab value in the lookback
                thr = {"GE30": (30, ">="), "GE25": (25, ">="), "GT5": (5.0, ">")}[k]
                def _latest(r, thr=thr):
                    lt = F.array_max(F.filter("EV", lambda x: x["D"].between(F.lit(r["START"]), F.lit(r["LAST_DAY"]))))
                    return (lt["V"] >= thr[0]) if thr[1] == ">=" else (lt["V"] > thr[0])
                m = _mask(rows, lambda r: F.coalesce(_latest(r), F.lit(False)))
            elif k in ("CUR", "STOP"):   # MRA current / stopped from the last fill end (any prescriber)
                def _cond(r, k=k):
                    last_end = F.array_max(F.transform(F.filter("EV", lambda x: x["D"] <= F.lit(r["LAST_DAY"])), lambda x: x["E"]))
                    c = (last_end >= F.date_sub(F.lit(r["LAST_DAY"]), p["MRA_CURRENT_GRACE_DAYS"])) if k == "CUR" \
                        else (F.date_add(last_end, p["MRA_STOP_GAP_DAYS"]) < F.lit(r["LAST_DAY"]))
                    return F.coalesce(c, F.lit(False))
                m = _mask(rows, _cond)
            exprs.append(F.struct(F.lit(f"{block}_{k}").alias("FLAG"), m.alias("MASK")))
        longs.append(sub.select("PATIENT_ID", F.explode(F.array(*exprs)).alias("_f")).select("PATIENT_ID", "_f.*"))
    flags = reduce(lambda a, b: a.unionByName(b), longs).filter(F.col("MASK") != 0)
    names = [f"{b}_{k}" for b, ks in PATIENT_MASKS.items() for k in ks]
    pf = flags.groupBy("PATIENT_ID").pivot("FLAG", names).agg(F.max("MASK")).fillna(0, subset=names)
    return _checkpoint(pf, p.get("TMP") and f"{p['TMP']}/_addon_pf")

def derive_patient_stems(pf, have_kbinder):
    """Bitwise (all 28 cohorts at once) combination of building blocks into the Section 13 patient sets."""
    c = F.col
    ALL = F.lit((1 << 28) - 1).cast("long")
    NOT = lambda x: x.bitwiseXOR(ALL)
    AND = lambda *xs: reduce(lambda a, b: a.bitwiseAND(b), xs)
    OR  = lambda *xs: reduce(lambda a, b: a.bitwiseOR(b), xs)
    dckd    = AND(c("T2D_LB"), OR(c("CKD_LB"), c("DKD_LB")), NOT(c("ESRD_LB")))   # diabetic CKD, not ESRD
    never_k = NOT(c("KER_EVER"))
    elig    = AND(dckd, c("ACEARB_R180"))          # T2D + CKD on ACE/ARB (label profile)
    s2      = AND(elig, never_k)
    s3      = AND(s2, c("UACR_GE30"))
    hfpef   = AND(c("HFPEF_LB"), NOT(c("HFREF_LB")), NOT(c("HFCOMB_LB")))
    stems = {
        # 13.1 untreated eligible pool (panel denominators)
        "S_DCKD": dckd, "S_UNTREATED": s2, "S_UNTREATED_UACR": s3, "S_UNTREATED_UACR_EGFR": AND(s3, c("EGFR_GE25")),
        "S_ELIG": elig, "S_ELIG_ON_K": AND(elig, c("KER_LB")),
        "S_ACE_ONLY": AND(dckd, c("ACEARB_R180"), NOT(c("SGLT2_R180")), NOT(c("GLP1_R180")), never_k),
        "S_SGLT2_GLP1_ONLY": AND(dckd, OR(c("SGLT2_R180"), c("GLP1_R180")), NOT(c("ACEARB_R180")), never_k),
        # 13.2 patient-level "new" events (HCP credit applied in the roll-up)
        "N_DKD": AND(c("DKD_D90"), NOT(c("DKD_BEF90"))),
        "N_SGLT2": AND(c("SGLT2_D90"), NOT(c("SGLT2_WASH"))),
        "N_ALBTEST": AND(c("ALBTEST_D90"), NOT(c("ALBTEST_WASH"))),
        # 13.3 HF with LVEF >= 40 % proxy
        "S_HFPEF": hfpef, "S_HFPEF_ECHO": AND(hfpef, c("ECHO_LB")),
        "S_HFPEF_LOOP_NOMRA": AND(hfpef, c("LOOP_R180"), NOT(c("MRA_R180")), NOT(c("KER_R180"))),
        "S_HFPEF_NEVER_K": AND(hfpef, never_k),
        # 13.4 potassium barrier (DKD denominators)
        "S_DKD_HYPERK": AND(dckd, OR(c("HYPERK_LB"), c("K_GT5"))),
        # 13.5 MRA switch (HCP credit applied in the roll-up)
        "M_CUR_DCKD": AND(c("MRA_CUR"), dckd), "M_STOP_DCKD": AND(c("MRA_STOP"), dckd), "M_STOP": c("MRA_STOP"),
    }
    if have_kbinder:
        stems["S_DKD_KBINDER"] = AND(dckd, c("KBINDER_R180"))
    out = pf.select("PATIENT_ID", *[v.alias(k) for k, v in stems.items()])
    return out.filter(reduce(lambda a, b: a | b, [F.col(k) != 0 for k in stems])), list(stems)

In [0]:
def build_addon_features(dx_full, rx_all, lab, px_all, rx_all_any_paid, hcp_universe, cohorts, rx_groups, p):
    rows, base = cohort_rows(cohorts)
    max_date = builtins.max(r["LAST_DAY"] for r in rows)
    rx_classes = rx_class_groups(rx_groups)
    have_kbinder = len(rx_classes["KBINDER"]) > 0
    ev = _checkpoint(build_block_events(dx_full, rx_all, lab, px_all, rx_classes, max_date),
                     p.get("TMP") and f"{p['TMP']}/_addon_ev")
    pf = build_patient_flags(ev, rows, p)
    ps, stem_keys = derive_patient_stems(pf, have_kbinder)
    ps = _checkpoint(ps, p.get("TMP") and f"{p['TMP']}/_addon_ps")

    # HCP panel: distinct patients with any paid Rx, Dx, lab or albuminuria-test claim from the HCP in the lookback
    def _pm(df):
        return (df.join(hcp_universe, "PROVIDER_ID", "left_semi")
                  .select("PROVIDER_ID", "PATIENT_ID",
                          F.months_between(F.trunc(F.to_date("SVC_DT"), "month"), F.lit(base)).cast("int").alias("M_IDX")))
    max_idx = builtins.max(r["E_IDX"] for r in rows)
    panel_src = reduce(lambda a, b: a.unionByName(b), [
        _pm(rx_all_any_paid), _pm(dx_full), _pm(lab),
        _pm(px_all.filter(F.col("PROCEDURE_CODE").isin("82043", "82044")))])
    hp = (panel_src.filter(F.col("PATIENT_ID").isNotNull() & F.col("M_IDX").between(0, max_idx))
                   .groupBy("PROVIDER_ID", "PATIENT_ID").agg(F.expr("bit_or(shiftleft(1L, M_IDX))").alias("MMASK")))
    hp = _checkpoint(hp, p.get("TMP") and f"{p['TMP']}/_addon_hp")
    lbm = {r["COHORT"]: builtins.sum(1 << i for i in range(r["S_IDX"], r["E_IDX"] + 1)) for r in rows}

    # HCP-credited events: trigger claims in the last 90 d (DKD Dx, paid SGLT2, albuminuria test) and the
    # HCP's own paid MRA claims in the lookback
    hpe = (ev.filter(F.col("BLOCK").isin("DKD", "SGLT2", "ALBTEST", "MRA"))
             .join(hcp_universe, "PROVIDER_ID", "left_semi")
             .groupBy("PROVIDER_ID", "PATIENT_ID", "BLOCK").agg(F.collect_set(F.struct("D")).alias("EV")))
    hpe = hpe.select("PROVIDER_ID", "PATIENT_ID", "BLOCK",
                     F.when(F.col("BLOCK") == "MRA", _mask(rows, lambda r: lambda_exists(*_win(r, "LB", p))))
                      .otherwise(_mask(rows, lambda r: lambda_exists(*_win(r, "D90", p)))).alias("HMASK"))
    hpe = (hpe.groupBy("PROVIDER_ID", "PATIENT_ID")
              .pivot("BLOCK", ["DKD", "SGLT2", "ALBTEST", "MRA"]).agg(F.max("HMASK")).fillna(0))
    hpe = _checkpoint(hpe, p.get("TMP") and f"{p['TMP']}/_addon_hpe")

    bit = lambda col, i: F.col(col).bitwiseAND(F.lit(1 << i)) != 0
    in_lb = lambda r: F.col("MMASK").bitwiseAND(F.lit(lbm[r["COHORT"]])) != 0
    panel_stems = [k for k in stem_keys if k.startswith("S_")]
    credited = {"N_DKD": "DKD", "N_SGLT2": "SGLT2", "N_ALBTEST": "ALBTEST",
                "M_CUR_DCKD": "MRA", "M_STOP_DCKD": "MRA", "M_STOP": "MRA"}
    joined = (hp.join(ps, "PATIENT_ID", "left").join(hpe, ["PROVIDER_ID", "PATIENT_ID"], "left")
                .fillna(0, subset=stem_keys + ["DKD", "SGLT2", "ALBTEST", "MRA"]))
    fields = ["PANEL", "MRA_PAT"] + stem_keys
    parts = []
    for batch in _chunks(list(enumerate(rows)), p.get("COHORT_BATCH", 7)):
        aggs = []
        for i, r in batch:
            cid = r["COHORT"]
            aggs.append(F.sum(F.when(in_lb(r), 1).otherwise(0)).alias(f"{cid}__PANEL"))
            aggs.append(F.sum(F.when(bit("MRA", i), 1).otherwise(0)).alias(f"{cid}__MRA_PAT"))
            for s in panel_stems:
                aggs.append(F.sum(F.when(in_lb(r) & bit(s, i), 1).otherwise(0)).alias(f"{cid}__{s}"))
            for s, hb in credited.items():
                aggs.append(F.sum(F.when(bit(hb, i) & bit(s, i), 1).otherwise(0)).alias(f"{cid}__{s}"))
        g = joined.groupBy("PROVIDER_ID").agg(*aggs)
        parts.append(_explode_cohorts(g, ["PROVIDER_ID"], [r for _, r in batch], fields).filter(F.col("PANEL") > 0))
    long = reduce(lambda a, b: a.unionByName(b), parts)

    # ---------------------------------------------------------------- naming (Section 13.1-13.5)
    pct = lambda num, den: F.when(F.col(den) > 0, F.col(num) / F.col(den))   # null when denominator is 0
    out = [F.col("PROVIDER_ID"), F.col("COHORT")]
    def num_pct(key, stem, den="PANEL"):
        out.append(F.col(key).alias(f"NUM_OF_PATIENTS_{stem}"))
        out.append(pct(key, den).alias(f"PCT_OF_PATIENTS_{stem}"))
    # 13.1
    num_pct("S_DCKD", "IN_HCP_PANEL_WITH_DIABETIC_CKD_IN_LAST_365_DAYS_PATIENT")
    num_pct("S_UNTREATED", "IN_HCP_PANEL_WITH_T2D_CKD_ON_ACE_ARB_NEVER_ON_KERENDIA_IN_LAST_365_DAYS_PATIENT")
    num_pct("S_UNTREATED_UACR", "IN_HCP_PANEL_WITH_T2D_CKD_ON_ACE_ARB_NEVER_ON_KERENDIA_AND_UACR_GE_30_IN_LAST_365_DAYS_PATIENT")
    num_pct("S_UNTREATED_UACR_EGFR", "IN_HCP_PANEL_WITH_T2D_CKD_ON_ACE_ARB_NEVER_ON_KERENDIA_AND_UACR_GE_30_AND_EGFR_GE_25_IN_LAST_365_DAYS_PATIENT")
    num_pct("S_ELIG", "IN_HCP_PANEL_ELIGIBLE_FOR_KERENDIA_IN_LAST_365_DAYS_PATIENT")
    num_pct("S_ELIG_ON_K", "IN_HCP_PANEL_ELIGIBLE_AND_ON_KERENDIA_IN_LAST_365_DAYS_PATIENT")
    out.append(pct("S_UNTREATED", "S_DCKD").alias("PCT_OF_DIABETIC_CKD_PATIENTS_IN_HCP_PANEL_UNTREATED_ELIGIBLE_IN_LAST_365_DAYS_PATIENT"))
    out.append(pct("S_ELIG_ON_K", "S_ELIG").alias("RATIO_KERENDIA_TREATED_TO_ELIGIBLE_PATIENTS_IN_HCP_PANEL_IN_LAST_365_DAYS_PATIENT"))
    num_pct("S_ACE_ONLY", "IN_HCP_PANEL_WITH_T2D_CKD_ON_ACE_ARB_ONLY_NEVER_ON_KERENDIA_IN_LAST_365_DAYS_PATIENT")
    num_pct("S_SGLT2_GLP1_ONLY", "IN_HCP_PANEL_WITH_T2D_CKD_ON_SGLT2_OR_GLP1_ONLY_NEVER_ON_KERENDIA_IN_LAST_365_DAYS_PATIENT")
    # 13.2
    num_pct("N_DKD", "THE_HCP_NEWLY_DIAGNOSED_WITH_DIABETIC_CKD_IN_LAST_90_DAYS_DX")
    num_pct("N_SGLT2", "THE_HCP_STARTED_ON_SGLT2_IN_LAST_90_DAYS_RX")
    num_pct("N_ALBTEST", "THE_HCP_ORDERED_NEW_ALBUMINURIA_TEST_IN_LAST_90_DAYS_LAB_DATA")
    # 13.3
    num_pct("S_HFPEF", "HFPEF_PROXY_IN_HCP_PANEL_IN_LAST_365_DAYS_PATIENT")
    num_pct("S_HFPEF_ECHO", "HFPEF_PROXY_WITH_ECHO_IN_HCP_PANEL_IN_LAST_365_DAYS_PATIENT")
    num_pct("S_HFPEF_LOOP_NOMRA", "HFPEF_PROXY_ON_LOOP_DIURETIC_NO_MRA_IN_HCP_PANEL_IN_LAST_365_DAYS_PATIENT")
    num_pct("S_HFPEF_NEVER_K", "HFPEF_PROXY_NEVER_ON_KERENDIA_IN_HCP_PANEL_IN_LAST_365_DAYS_PATIENT")
    # 13.4 (denominator = DKD patients in the panel)
    out.append(F.col("S_DKD_HYPERK").alias("NUM_OF_DKD_PATIENTS_IN_HCP_PANEL_WITH_HYPERKALEMIA_IN_LAST_365_DAYS_PATIENT"))
    out.append(pct("S_DKD_HYPERK", "S_DCKD").alias("PCT_OF_DKD_PATIENTS_IN_HCP_PANEL_WITH_HYPERKALEMIA_IN_LAST_365_DAYS_PATIENT"))
    if have_kbinder:
        out.append(F.col("S_DKD_KBINDER").alias("NUM_OF_DKD_PATIENTS_IN_HCP_PANEL_ON_POTASSIUM_BINDER_IN_LAST_365_DAYS_PATIENT"))
        out.append(pct("S_DKD_KBINDER", "S_DCKD").alias("PCT_OF_DKD_PATIENTS_IN_HCP_PANEL_ON_POTASSIUM_BINDER_IN_LAST_365_DAYS_PATIENT"))
    # 13.5 (THE_HCP_PRESCRIBED_WITH_STEROIDAL_MRA_IN_LAST_365_DAYS_RX intentionally NOT built: NBA 3.1 removal note)
    num_pct("M_CUR_DCKD", "THE_HCP_PRESCRIBED_WITH_STEROIDAL_MRA_CURRENTLY_ON_MRA_WITH_T2D_CKD_IN_LAST_365_DAYS_RX")
    num_pct("M_STOP_DCKD", "THE_HCP_PRESCRIBED_WITH_STEROIDAL_MRA_THAT_STOPPED_MRA_WITH_T2D_CKD_IN_LAST_365_DAYS_RX")
    out.append(pct("M_STOP", "MRA_PAT").alias("PCT_OF_STEROIDAL_MRA_PATIENTS_THE_HCP_PRESCRIBED_THAT_STOPPED_MRA_IN_LAST_365_DAYS_RX"))
    out.append(F.col("PANEL").alias("HCP_PANEL_SIZE_IN_LAST_365_DAYS_PATIENT"))

    feats = (long.select(*out).join(_anchor_df(rows), "COHORT")
                 .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")))
    return feats, rx_classes, have_kbinder

In [0]:
# Patient-level status uses claims from ALL providers (Section 13.0) -> inputs are NOT filtered to the universe here;
# the universe filter is applied to the HCP panel and to HCP-credited claims inside build_addon_features.
dx_full = spark.read.format("delta").load(DX_FULL_PATH)
rx_all  = spark.read.format("delta").load(RX_ALL_PATH)
lab     = spark.read.format("delta").load(LAB_PATH)
px_all  = spark.read.format("delta").load(PX_ALL_PATH)

addon, rx_classes, have_kbinder = build_addon_features(
    dx_full, rx_all, lab, px_all, rx_all.filter(F.col("CLAIM_STATUS") == "S"),
    hcp_universe, cohorts, rx_event_keys, ADDON_PARAMS)

for k, v in rx_classes.items():
    print(f"{k:8s} ({len(v):2d}): {v}")
if not have_kbinder:
    print("WARNING: no Lokelma/Veltassa in the Rx event list -> potassium-binder stem (13.4) not built")

(addon.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .partitionBy("COHORT").save(ADDON_OUT))
print(f"Written: {ADDON_OUT} | features: {len([c for c in addon.columns if c.startswith(('NUM_', 'PCT_', 'RATIO_'))])}")

In [0]:
df = spark.read.format("delta").load(ADDON_OUT)
cohort_ids = {r[0] for r in df.select("COHORT").distinct().collect()}
n_outside  = df.select("PROVIDER_ID").distinct().join(hcp_universe_df, "PROVIDER_ID", "left_anti").count()
n_dupes    = df.groupBy("HCP_COHORT_ID").count().filter("count > 1").count()
print(f"Add-on: {df.count():,} rows | cohorts={len(cohort_ids)}/28 | outside universe={n_outside} | duplicate HCP_COHORT_ID={n_dupes}")
assert n_outside == 0 and n_dupes == 0

# Sanity: PCT in [0, 1]; NUM <= panel
feat_pct = [c for c in df.columns if c.startswith(("PCT_", "RATIO_"))]
bad = df.select(*[F.sum(((F.col(c) < 0) | (F.col(c) > 1)).cast("int")).alias(c) for c in feat_pct]).first().asDict()
print("PCT/RATIO outside [0,1]:", {k: v for k, v in bad.items() if v} or "none")

# Distribution vs Section 13 reference medians (c28 = latest cohort)
num_cols = [c for c in df.columns if c.startswith("NUM_")]
nz = df.filter(F.col("COHORT") == "c28").select(*[F.avg((F.col(c) > 0).cast("double")).alias(c) for c in num_cols]).first().asDict()
for c in num_cols:
    print(f"{nz[c]:6.1%} non-zero  {c}")

dbutils.fs.rm(TMP_BASE, True)   # drop intermediate checkpoints
dbutils.notebook.exit("Add-on patient features written")